In [7]:
import math
import urllib.request
from collections import defaultdict,Counter
TRAIN_URL = (
    "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/"
    "master/en_ewt-ud-train.conllu"
)

TEST_URL = (
    "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/"
    "master/en_ewt-ud-test.conllu"
)
def download_file(url,filename):
    try:
        urllib.request.urlretrieve(url,filename)
        print(f"Downloaded:{filename}")
    except Exception as e:
        print("Error downloading file:",e)
download_file(TRAIN_URL,"en_ewt-ud-train.conllu")
download_file(TEST_URL,"en_ewt-ud-test.conllu")
def load_conllu(filename):
    sentences=[]
    words=[]
    tags=[]
    with open(filename,"r",encoding="utf-8") as file:
        for line in file:
            line=line.strip()
            if not line:
                if words:
                    sentences.append((words,tags))
                    words=[]
                    tags=[]
                continue
            if line.startswith("#"):
                continue
            columns=line.split("\t")
            if len(columns)!=10:
                continue
            token_id=columns[0]
            if "-" in token_id or "." in token_id:
                continue
            word=columns[1]
            pos_tag=columns[3]
            words.append(word)
            tags.append(pos_tag)
        if words:
            sentences.append((words,tags))
        return sentences

train_data = load_conllu("en_ewt-ud-train.conllu")
test_data = load_conllu("en_ewt-ud-test.conllu")
print("\nTraining sentences:",len(train_data))
print("Testing sentences:",len(test_data))
transition_counts=defaultdict(Counter)
emission_counts=defaultdict(Counter)
tag_counts=Counter()
START="<START>"
END="<END>"
for words,tags in train_data:
    previous_tag=START
    for word,tag in zip(words,tags):
        transition_counts[previous_tag][tag]+=1
        emission_counts[tag][word.lower()]+=1
        tag_counts[tag]+=1
        previous_tag=tag
    transition_counts[previous_tag][END]+=1
tags_list=list(tag_counts.keys())
print("\nPOS Tags:")
print(tags_list)
def transition_probability(previous_tag,current_tag):
    numerator=transition_counts[previous_tag][current_tag]+2
    denominator=(sum(transition_counts[previous_tag].values())+len(tags_list)+1)
    return numerator/denominator
def emission_probability(tag,word):
    word=word.lower()
    vocabulary_size=len(emission_counts[tag])+1
    numerator=emission_counts[tag][word]+1
    denominator=tag_counts[tag]+vocabulary_size
    return numerator/denominator
def viterbi(words):
    words=[word.lower() for word in words]
    viterbi_table=[]
    backpointer=[]
    first_column={}
    first_backpointer={}
    for tag in tags_list:
        transition_prob=transition_probability(START,tag)
        emission_prob=emission_probability(tag,words[0])
        first_column[tag]=(math.log(transition_prob)+math.log(emission_prob))
        first_backpointer[tag]=None
    viterbi_table.append(first_column)
    backpointer.append(first_backpointer)
    for i in range(1,len(words)):
        current_column={}
        current_backpointer={}
        word=words[i]
        for current_tag in tags_list:
            best_probability=float("-inf")
            best_previous_tag=None
            emission_prob=emission_probability(current_tag,word)
            for previous_tag in tags_list:
                transition_prob=transition_probability(previous_tag,current_tag)
                probability=(viterbi_table[i-1][previous_tag]+math.log(transition_prob)+math.log(emission_prob))
                if probability>best_probability:
                    best_probability=probability
                    best_previous_tag=previous_tag
            current_column[current_tag]=best_probability
            current_backpointer[current_tag]=best_previous_tag
        viterbi_table.append(current_column)
        backpointer.append(current_backpointer)
    best_final_probability=float("-inf")
    best_final_tag=None
    for tag in tags_list:
        probability=(viterbi_table[-1][tag]+math.log(transition_probability(tag,END)))
        if probability>best_final_probability:
              best_final_probability=probability
              best_final_tag=tag
    best_tags=[best_final_tag]
    for i in range(len(words)-1,0,-1):
        previous_tag=backpointer[i][best_tags[-1]]
        best_tags.append(previous_tag)
    best_tags.reverse()
    return best_tags
sentence=input("\nEnter sentence:")
words=sentence.split()
predicted_tags=viterbi(words)
print("\nPredicted POS Tags:")
for word,tag in zip(words,predicted_tags):
    print(f"{word}-->{tag}")
correct=0
total=0
print("\nEvaluvating on test dataset")
for words,actual_tags in test_data:
    predicted_tags=viterbi(words)
    for predicted,actual in zip(predicted_tags,actual_tags):
        if predicted==actual:
            correct+=1
        total+=1
accuracy=(correct/total)*100
print("\n========Evaluation Report===========")
print("Total test tokens:",total)
print("Correct Predictions:",correct)
print("Incorrect Predictions:",total-correct)
print(f"POS Tagging Accuracy:{accuracy:.2f}%")
print("==================")
tag_correct=Counter()
tag_total=Counter()
for words,actual_tags in test_data:
    predicted_tags=viterbi(words)
    for predicted,actual in zip(predicted_tags,actual_tags):
        tag_total[actual]+=1
        if predicted==actual:
            tag_correct[actual]+=1
print("\nPer-POS Tag Accuracy:")
print("---------------------")
for tag in sorted(tag_total):
    tag_accuracy=(tag_correct[tag]/tag_total[tag])*100
    print(f"{tag:8s}:" f"{tag_accuracy:6.2f}%" f"({tag_correct[tag]}/{tag_total[tag]})")
    
        
    
        



Downloaded:en_ewt-ud-train.conllu
Downloaded:en_ewt-ud-test.conllu

Training sentences: 12544
Testing sentences: 2077

POS Tags:
['PROPN', 'PUNCT', 'ADJ', 'NOUN', 'VERB', 'DET', 'ADP', 'AUX', 'PRON', 'PART', 'SCONJ', 'NUM', 'ADV', 'CCONJ', 'INTJ', 'X', 'SYM']



Enter sentence: The student reads a book



Predicted POS Tags:
The-->DET
student-->NOUN
reads-->ADP
a-->DET
book-->NOUN

Evaluvating on test dataset

========Evaluation Report===========
Total test tokens: 25094
Correct Predictions: 16625
Incorrect Predictions: 8469
POS Tagging Accuracy:66.25%

Per-POS Tag Accuracy:
---------------------
ADJ     : 55.37%(990/1788)
ADP     : 73.14%(1481/2025)
ADV     : 67.25%(801/1191)
AUX     : 86.84%(1340/1543)
CCONJ   : 71.88%(529/736)
DET     : 86.77%(1646/1897)
INTJ    : 81.82%(99/121)
NOUN    : 51.35%(2117/4123)
NUM     : 33.95%(184/542)
PART    : 82.74%(537/649)
PRON    : 89.74%(1942/2164)
PROPN   : 20.05%(416/2075)
PUNCT   : 80.10%(2480/3096)
SCONJ   : 63.28%(243/384)
SYM     : 43.36%(49/113)
VERB    : 66.83%(1741/2605)
X       : 71.43%(30/42)
